In [1]:
import requests

In [2]:
import requests

repo_owner = 'evidentlyai'
repo_name = 'docs'
branch_name = 'main'
zip_url = f'https://github.com/{repo_owner}/{repo_name}/archive/refs/heads/{branch_name}.zip'
zip_response = requests.get(zip_url)

In [3]:
len(zip_response.content)

17545668

In [4]:
import io
import zipfile

zip_archive = zipfile.ZipFile(io.BytesIO(zip_response.content))

In [5]:
filenames = zip_archive.namelist()
filenames[20:30]

['docs-main/docs/library/report.mdx',
 'docs-main/docs/library/synthetic_data_api.mdx',
 'docs-main/docs/library/tags_metadata.mdx',
 'docs-main/docs/library/tests.mdx',
 'docs-main/docs/platform/',
 'docs-main/docs/platform/alerts.mdx',
 'docs-main/docs/platform/dashboard_add_panels.mdx',
 'docs-main/docs/platform/dashboard_add_panels_ui.mdx',
 'docs-main/docs/platform/dashboard_overview.mdx',
 'docs-main/docs/platform/dashboard_panel_types.mdx']

In [6]:
filename = 'docs-main/docs/platform/alerts.mdx'
mdx_file = zip_archive.open(filename)
mdx_content = mdx_file.read().decode('utf-8')
print(mdx_content[:150])


---
title: 'Alerts'
description: 'How to set up alerts.'
---

<Check>
  Built-in alerting is a Pro feature available in the **Evidently Cloud** and **


In [7]:
import frontmatter

post = frontmatter.loads(mdx_content)
print(post.content[:100])

<Check>
  Built-in alerting is a Pro feature available in the **Evidently Cloud** and **Evidently En


In [8]:
post.metadata

{'title': 'Alerts', 'description': 'How to set up alerts.'}

In [9]:
filename_corrected = filename.split('/', 1)[-1]
print(filename_corrected)

docs/platform/alerts.mdx


In [10]:
doc = {
    'content': post.content,
    'title': post.metadata.get('title'),
    'description': post.metadata.get('description'),
    'filename': filename_corrected
}

In [11]:
def read_github_repository(repo_owner, repo_name, branch="main"):
    url = f"https://github.com/{repo_owner}/{repo_name}/archive/refs/heads/{branch}.zip"
    response = requests.get(url)
    response.raise_for_status()

    documents = []
    with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
        for file_path in zip_ref.namelist():
            if not file_path.endswith(('.md', '.mdx')):
                continue
            with zip_ref.open(file_path) as file:
                content = file.read().decode('utf-8')
                post = frontmatter.loads(content)
                doc = {
                    'content': post.content,
                    'title': post.metadata.get('title'),
                    'description': post.metadata.get('description'),
                    'filename': file_path.split('/', 1)[-1]
                }
                documents.append(doc)

    return documents

In [12]:
repo_owner = 'evidentlyai'
repo_name = 'docs'

documents = read_github_repository(repo_owner, repo_name)

print(f"Downloaded {len(documents)} documents")

Downloaded 95 documents


In [13]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)

files = reader.read()

print(f"Loaded {len(files)} documents")

Loaded 95 documents


In [14]:
document = files[10]

print(document.filename)
print(document.content[:160])

docs/library/output_formats.mdx
---
title: 'Output formats'
description: 'How to export the evaluation results.'
---

You can view or export Reports in multiple formats.

**Pre-requisites**:




In [15]:
import frontmatter 

post = frontmatter.loads(document.content)
data = post.to_dict()
data['filename'] = document.filename

In [16]:

documents = [f.parse() for f in files]

In [17]:
from minsearch import Index

In [18]:
documents[10]

{'title': 'Output formats',
 'description': 'How to export the evaluation results.',
 'content': 'You can view or export Reports in multiple formats.\n\n**Pre-requisites**:\n\n* You know how to [generate Reports](/docs/library/report).\n\n## Log to Workspace\n\nYou can save the computed Report in Evidently Cloud or your local workspace.\n\n```python\nws.add_run(project.id, my_eval, include_data=False)\n```\n\n<Info>\n  **Uploading evals**. Check Quickstart examples [for ML](/quickstart_ml) or [for LLM](/quickstart_llm) for a full workflow.\n</Info>\n\n## View in Jupyter notebook\n\nYou can directly render the visual summary of evaluation results in interactive Python environments like Jupyter notebook or Colab.\n\nAfter running the Report, simply call the resulting Python object:\n\n```python\nmy_report\n```\n\nThis will render the HTML object directly in the notebook cell.\n\n## HTML\n\nYou can also save this interactive visual Report as an HTML file to open in a browser:\n\n```python

## Search

In [19]:
from minsearch import Index

In [20]:
index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(documents)

In [21]:
query = 'LLM as a Judge'
results = index.search(query, num_results=5)
print(results)


[{'title': 'LLM as a judge', 'description': 'How to create and evaluate an LLM judge.', 'content': 'import CloudSignup from \'/snippets/cloud_signup.mdx\';\nimport CreateProject from \'/snippets/create_project.mdx\';\n\nIn this tutorial, we\'ll show how to evaluate text for custom criteria using LLM as the judge, and evaluate the LLM judge itself.\n\n<Info>\n  **This is a local example.** You will run and explore results using the open-source Python library. At the end, we’ll optionally show how to upload results to the Evidently Platform for easy exploration.\n</Info>\n\nWe\'ll explore two ways to use an LLM as a judge:\n\n- **Reference-based**. Compare new responses against a reference. This is useful for regression testing or whenever you have a "ground truth" (approved responses) to compare against.\n- **Open-ended**. Evaluate responses based on custom criteria, which helps evaluate new outputs when there\'s no reference available.\n\nWe will focus on demonstrating **how to create 

## Chunking

In [22]:
doc_sizes = [(doc.filename, len(doc.content)) for doc in files]
doc_sizes.sort(key=lambda x: x[1], reverse=True)

for filename, size in doc_sizes[:5]:
    print(f"{filename}: {size} characters")


metrics/all_metrics.mdx: 55085 characters
metrics/all_descriptors.mdx: 31976 characters
docs/platform/dashboard_panel_types.mdx: 31647 characters
docs/library/leftover_content.mdx: 28742 characters
metrics/customize_llm_judge.mdx: 26847 characters


In [23]:
document = list(range(0, 100))

document


[0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 82,
 83,
 84,
 85,
 86,
 87,
 88,
 89,
 90,
 91,
 92,
 93,
 94,
 95,
 96,
 97,
 98,
 99]

In [24]:
window_size = 10
start = 0
step = 5

chunks = []

while start < len(document):
    end = start + window_size
    chunk = document[start:end]
    if len(chunk) < window_size:
        break
    chunks.append(chunk)
    print(chunk)
    start = start + step

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
[10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
[15, 16, 17, 18, 19, 20, 21, 22, 23, 24]
[20, 21, 22, 23, 24, 25, 26, 27, 28, 29]
[25, 26, 27, 28, 29, 30, 31, 32, 33, 34]
[30, 31, 32, 33, 34, 35, 36, 37, 38, 39]
[35, 36, 37, 38, 39, 40, 41, 42, 43, 44]
[40, 41, 42, 43, 44, 45, 46, 47, 48, 49]
[45, 46, 47, 48, 49, 50, 51, 52, 53, 54]
[50, 51, 52, 53, 54, 55, 56, 57, 58, 59]
[55, 56, 57, 58, 59, 60, 61, 62, 63, 64]
[60, 61, 62, 63, 64, 65, 66, 67, 68, 69]
[65, 66, 67, 68, 69, 70, 71, 72, 73, 74]
[70, 71, 72, 73, 74, 75, 76, 77, 78, 79]
[75, 76, 77, 78, 79, 80, 81, 82, 83, 84]
[80, 81, 82, 83, 84, 85, 86, 87, 88, 89]
[85, 86, 87, 88, 89, 90, 91, 92, 93, 94]
[90, 91, 92, 93, 94, 95, 96, 97, 98, 99]


In [25]:
def sliding_window(text, size=1000, step=500):
    chunks = []
    start = 0
    text_length = len(text)

    while start < text_length:
        end = start + size
        chunk = text[start:end]
        chunks.append({'start': start, 'content': chunk})

        start = end - step

        if end >= text_length:
            break

    return chunks

In [26]:
sliding_window(results[0]['content'])

[{'start': 0,
  'content': 'import CloudSignup from \'/snippets/cloud_signup.mdx\';\nimport CreateProject from \'/snippets/create_project.mdx\';\n\nIn this tutorial, we\'ll show how to evaluate text for custom criteria using LLM as the judge, and evaluate the LLM judge itself.\n\n<Info>\n  **This is a local example.** You will run and explore results using the open-source Python library. At the end, we’ll optionally show how to upload results to the Evidently Platform for easy exploration.\n</Info>\n\nWe\'ll explore two ways to use an LLM as a judge:\n\n- **Reference-based**. Compare new responses against a reference. This is useful for regression testing or whenever you have a "ground truth" (approved responses) to compare against.\n- **Open-ended**. Evaluate responses based on custom criteria, which helps evaluate new outputs when there\'s no reference available.\n\nWe will focus on demonstrating **how to create and tune the LLM evaluator**, which you can then apply in different cont

In [27]:
len(sliding_window(results[0]['content']))

43

In [28]:
documents[30]

{'title': 'Overview',
 'description': 'How production AI quality monitoring works.',
 'content': 'AI observability lets you evaluate the quality of the inputs and outputs of your AI application as it runs in production. This gives an up-to-date view of your system behavior and helps spot and fix issues.\n\nEvidently offers several ways to set up monitoring.\n\n## Batch monitoring jobs\n\n<Check>\n  Supported in: `Evidently OSS`, `Evidently Cloud` and `Evidently Enterprise`.\n</Check>\n\n**Best for**: batch ML pipelines, regression testing, and near real-time ML systems that don’t need instant quality evaluations.\n\n![](/images/monitoring_flow_batch.png)\n\n**How it works**:\n\n* **Build your evaluation pipeline**. Create a pipeline in your infrastructure to run monitoring jobs. This can be a Python script, cron job, or orchestrated with a tool like Airflow. Run it at regular intervals (e.g., hourly, daily) or trigger it when new data or labels arrive.\n\n* **Run metric calculations**.

In [29]:
document_chunks = []

for doc in documents:
    if not doc.get('content'):
        continue
    copy = doc.copy()
    content = copy.pop('content')

    chunks = sliding_window(content, size=3000, step=1500)

    for i, chunk in enumerate(chunks):
        chunk.update(copy)
        chunk['chunk_id'] = i
        document_chunks.append(chunk)

In [30]:
document_chunks[10]

{'start': 9000,
 'content': 'cation=[BinaryClassification(\n        target="target",\n        prediction_labels="prediction")],\n    categorical_columns=["target", "prediction"])\n```\n\nAvailable options and defaults:\n\n```python\n    target: str = "target"\n    prediction_labels: Optional[str] = None\n    prediction_probas: Optional[str] = "prediction" #if probabilistic classification\n    pos_label: Label = 1 #name of the positive label\n    labels: Optional[Dict[Label, str]] = None\n```\n\n### Ranking\n\n#### RecSys\n\nTo evaluate recommender systems performance, you must map the columns with:\n\n- Prediction: this could be predicted score or rank.\n- Target: relevance labels (e.g., this could be an interaction result like user click or upvote, or a true relevance label)\n\nThe **target** column can contain either:\n\n- a binary label (where `1` is a positive outcome)\n- any scores (positive values, where a higher value corresponds to a better match or a more valuable user action)

In [31]:
chunk_index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
chunk_index.fit(document_chunks)

In [32]:
results = chunk_index.search(query)

In [33]:
results

[{'start': 0,
  'content': 'import CloudSignup from \'/snippets/cloud_signup.mdx\';\nimport CreateProject from \'/snippets/create_project.mdx\';\n\nIn this tutorial, we\'ll show how to evaluate text for custom criteria using LLM as the judge, and evaluate the LLM judge itself.\n\n<Info>\n  **This is a local example.** You will run and explore results using the open-source Python library. At the end, we’ll optionally show how to upload results to the Evidently Platform for easy exploration.\n</Info>\n\nWe\'ll explore two ways to use an LLM as a judge:\n\n- **Reference-based**. Compare new responses against a reference. This is useful for regression testing or whenever you have a "ground truth" (approved responses) to compare against.\n- **Open-ended**. Evaluate responses based on custom criteria, which helps evaluate new outputs when there\'s no reference available.\n\nWe will focus on demonstrating **how to create and tune the LLM evaluator**, which you can then apply in different cont

In [34]:
from gitsource import chunk_documents

document_chunks = chunk_documents(documents, size=3000, step=1500)

## RAG

In [35]:
from openai import OpenAI

openai_client = OpenAI()

In [36]:
search_result = chunk_index.search(query, num_results=5)

In [37]:
query = 'how do I implement llm as a judge?'

In [38]:
import json

search_result_json = json.dumps(search_result, indent=2)

In [39]:
instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

user_prompt = f"""
<QUESTION>
{query}
</QUESTION>

<CONTEXT>
{search_result_json}
</CONTEXT>
""".strip()

In [40]:
def llm(user_prompt, instructions=None, model='gpt-4o-mini'):

    messages = []

    if instructions is not None:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text


In [41]:
answer = llm(user_prompt, instructions)

In [42]:
answer

'To implement an LLM (Large Language Model) as a judge, follow these steps based on the provided context:\n\n1. **Setup Your Environment**:\n   - Ensure you have basic Python knowledge.\n   - Install the required library (Evidently):\n     ```python\n     pip install evidently\n     ```\n   - Import the necessary modules:\n     ```python\n     import pandas as pd\n     import numpy as np\n     from evidently import Dataset, Report\n     from evidently.llm.templates import BinaryClassificationPromptTemplate\n     ```\n\n2. **Obtain API Key**:\n   - Get your OpenAI API key and set it as an environment variable:\n     ```python\n     import os\n     os.environ["OPENAI_API_KEY"] = "YOUR_KEY"\n     ```\n\n3. **Create Your Evaluation Dataset**:\n   - Construct a toy Q&A dataset with questions, target responses, new responses, and manually labeled correctness.\n   - Example: \n     ```python\n     data = [\n         ["Hi there, how do I reset my password?", "To reset your password, click on .

In [43]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

In [44]:
def search(query):
    return chunk_index.search(query, num_results=5)


In [45]:
instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

def build_prompt(query, search_results):
    search_result_json = json.dumps(search_results, indent=2)

    user_prompt = f"""
    <QUESTION>
    {query}
    </QUESTION>

    <CONTEXT>
    {search_result_json}
    </CONTEXT>
    """.strip()

    return user_prompt


In [46]:
rag('how do I implement llm as a judge?')


'To implement an LLM as a judge, follow these steps:\n\n1. **Understand Evaluation Approaches**:\n   - You can use a **Reference-based** approach, where new responses are compared against approved responses, or an **Open-ended** approach, where responses are evaluated based on custom criteria when no reference is available.\n\n2. **Set Up the Environment**:\n   - You will need basic Python knowledge and an OpenAI API key. \n   - Install the necessary library by running:\n     ```python\n     pip install evidently\n     ```\n   - Import the required modules:\n     ```python\n     import pandas as pd\n     import numpy as np\n     from evidently import Dataset, DataDefinition, Report, BinaryClassification\n     from evidently.llm.templates import BinaryClassificationPromptTemplate\n     ```\n\n3. **Create the Evaluation Dataset**:\n   - Develop a toy Q&A dataset with questions, approved target responses, new responses, and manual labels. This helps in formulating better criteria and gene

## Structured Output 

In [48]:
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

print(f"Indexed {len(chunked_docs)} chunks from {len(files)} documents")


Indexed 385 chunks from 95 documents


In [49]:
def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

In [50]:
import json

instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return prompt_template.format(
        question=question,
        context=context
    )


In [51]:
def llm(user_prompt, instructions=None, model="gpt-4o-mini"):
    messages = []

    if instructions:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

In [52]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm(prompt, instructions)

In [53]:
answer = rag('how do I implement LLM as a judge?')

In [54]:
answer

'To implement an LLM as a judge, follow these steps from the tutorial:\n\n1. **Install Necessary Libraries**:\n   Install the Evidently library using:\n   ```python\n   pip install evidently\n   ```\n\n2. **Import Required Modules**:\n   Import the necessary modules in your script:\n   ```python\n   import pandas as pd\n   import numpy as np\n   from evidently import Dataset\n   from evidently import DataDefinition\n   from evidently import Report\n   from evidently import BinaryClassification\n   from evidently.presets import TextEvals\n   from evidently.llm.templates import BinaryClassificationPromptTemplate\n   ```\n\n3. **Set Up the OpenAI API Key**:\n   Pass your OpenAI API key as an environment variable:\n   ```python\n   import os\n   os.environ["OPENAI_API_KEY"] = "YOUR_KEY"\n   ```\n\n4. **Create the Evaluation Dataset**:\n   - Generate a toy Q&A dataset that includes Questions, Target responses, New responses, and Manual labels to indicate correctness.\n   - For example:\n   

In [60]:
def llm_structured(
    user_prompt,
    output_type,
    instructions=None,
    model="gpt-4o-mini",
):
    messages = []

    if instructions:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.parse(
        model=model,
        input=messages,
        text_format=output_type
    )

    return response.output_parsed

In [61]:
from pydantic import BaseModel

class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]

In [62]:
response = llm_structured(
    instructions="Extract the event information.",
    user_prompt="Alice and Bob are going to a science fair on Friday.",
    output_type=CalendarEvent,
)

In [65]:
class RAGResponse(BaseModel):
    answer: str
    found_answer: bool

In [66]:
RAGResponse.model_json_schema()

{'properties': {'answer': {'title': 'Answer', 'type': 'string'},
  'found_answer': {'title': 'Found Answer', 'type': 'boolean'}},
 'required': ['answer', 'found_answer'],
 'title': 'RAGResponse',
 'type': 'object'}

In [67]:
def rag_structured(query, output_type=RAGResponse):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm_structured(
        instructions=instructions,
        user_prompt=prompt,
        output_type=output_type,
    )


In [68]:
answer = rag_structured('how do I do llm evals?')

print(answer.answer[:100])
print(answer.found_answer)


To perform LLM evaluations, you can follow these steps based on the provided context:

1. **Installa
True


In [80]:
answer = rag_structured('how do I install kafka on windows?')

print(answer.answer[:100])
print(answer.found_answer)


False


In [70]:
from typing import Optional

class RAGResponse(BaseModel):
    answer: Optional[str] = None
    found_answer: bool

In [71]:
RAGResponse.model_json_schema()

{'properties': {'answer': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'title': 'Answer'},
  'found_answer': {'title': 'Found Answer', 'type': 'boolean'}},
 'required': ['found_answer'],
 'title': 'RAGResponse',
 'type': 'object'}

In [77]:
answer = rag_structured('how do I install kafka on windows?')

print(answer.answer[:100])
print(answer.found_answer)


False


In [73]:
instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.

If you don't find the answer, set `answer` to None
"""

In [75]:
class RAGResponse(BaseModel):
    """
    The response from the documentation RAG system

    If the answer to the question wasn't found in the database, `answer` is None
    """
    answer: Optional[str] = None
    found_answer: bool

In [76]:
RAGResponse.model_json_schema()

{'description': "The response from the documentation RAG system\n\nIf the answer to the question wasn't found in the database, `answer` is None",
 'properties': {'answer': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'title': 'Answer'},
  'found_answer': {'title': 'Found Answer', 'type': 'boolean'}},
 'required': ['found_answer'],
 'title': 'RAGResponse',
 'type': 'object'}

In [78]:
from pydantic import Field

class RAGResponse(BaseModel):
    """
    The response from the documentation RAG system
    """
    answer: Optional[str] = Field(None, description="Answer to the question or None if it's not found")
    found_answer: bool = Field(description="True if the answer is found, False otherwise")

In [79]:
RAGResponse.model_json_schema()

{'description': 'The response from the documentation RAG system',
 'properties': {'answer': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'description': "Answer to the question or None if it's not found",
   'title': 'Answer'},
  'found_answer': {'description': 'True if the answer is found, False otherwise',
   'title': 'Found Answer',
   'type': 'boolean'}},
 'required': ['found_answer'],
 'title': 'RAGResponse',
 'type': 'object'}

In [81]:
from typing import Literal

class RAGResponse(BaseModel):
    """
    This model provides a structured answer with metadata about the response,
    including confidence, categorization, and follow-up suggestions.
    """

    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how certain the answer is")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user might want to ask")

In [82]:
RAGResponse.model_json_schema()

{'description': 'This model provides a structured answer with metadata about the response,\nincluding confidence, categorization, and follow-up suggestions.',
 'properties': {'answer': {'description': "The main answer to the user's question in markdown",
   'title': 'Answer',
   'type': 'string'},
  'found_answer': {'description': 'True if relevant information was found in the documentation',
   'title': 'Found Answer',
   'type': 'boolean'},
  'confidence': {'description': 'Confidence score from 0.0 to 1.0 indicating how certain the answer is',
   'title': 'Confidence',
   'type': 'number'},
  'confidence_explanation': {'description': 'Explanation about the confidence level',
   'title': 'Confidence Explanation',
   'type': 'string'},
  'answer_type': {'description': 'The category of the answer',
   'enum': ['how-to',
    'explanation',
    'troubleshooting',
    'comparison',
    'reference'],
   'title': 'Answer Type',
   'type': 'string'},
  'followup_questions': {'description': 'S

In [83]:
answer = rag_structured('how do I evaluate llms', RAGResponse)

In [89]:
answer

RAGResponse(answer='To evaluate LLMs effectively, you can follow these steps outlined in the provided documentation:\n\n### 1. **Set Up Evaluator LLMs**  \n   - Install the necessary libraries:  \n   ```python  \n   pip install evidently litellm  \n   ```  \n   - Import the required components, including Dataset, DataDefinition, and Report from the `evidently` library.\n   - Set the API keys for your LLMs:  \n   ```python  \n   import os  \n   os.environ["OPENAI_API_KEY"] = "YOUR KEY"  \n   os.environ["GEMINI_API_KEY"] = "YOUR KEY"  \n   os.environ["ANTHROPIC_API_KEY"] = "YOUR KEY"  \n   ```\n\n### 2. **Define Evaluation Prompts**  \n   - Create a prompt for judging the output\'s appropriateness using `BinaryClassificationPromptTemplate`. This should highlight what constitutes appropriate and inappropriate content.\n   ```python  \n   us_corp_email_appropriateness = BinaryClassificationPromptTemplate(  \n       pre_messages=[  \n           ("system", "You are an expert in U.S. corporat

In [90]:
answer = rag_structured('how do I install kafka on windows?', RAGResponse)

In [94]:
answer

RAGResponse(answer='', found_answer=False, confidence=0.0, confidence_explanation='The context provided does not contain any information regarding the installation of Kafka on Windows.', answer_type='reference', followup_questions=['What are the steps to configure Kafka after installation?', 'Can I run Kafka on other operating systems?', 'Where can I find Kafka documentation?'])

In [ ]:
messages = [
    {"role": "user", "content": "tell me a bad time story about a unicorn"}
]

stream = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
    stream=True
)

response = None
for event in stream:
    if hasattr(event, 'delta'):
        print(event.delta, end='')
    if hasattr(event, 'response'):
        response = event.response


Once upon a time, in a hidden valley, there lived a unicorn named Sparkle. Unlike other unicorns who shone brightly and spread joy, Sparkle was known for his rather gloomy demeanor. His mane was a dull gray, his horn lacked the shimmering luster of his friends, and he often wandered off alone, avoiding the other creatures of the forest.

One stormy night, a fierce storm rolled into the valley. The winds howled, and the rain poured down in sheets. The other animals huddled together for warmth and comfort, but Sparkle was too absorbed in his own sadness to notice. He wandered deeper into the woods where trees whispered secrets of despair, and shadows danced menacingly.

In his solitude, Sparkle stumbled upon a dark cave. Curious and desperate for an escape from the storm, he entered, only to find it was already occupied by a grumpy troll named Grumble. Grumble had been cursed to guard a treasure but was eternally unhappy about it. When Sparkle entered, the troll grunted, “What do you wan

In [99]:
from pydantic import BaseModel, Field

class ArticleSection(BaseModel):
    header: str
    text: str = Field(description="text of the section in markdown")

class ArticleResponse(BaseModel):
    title: str
    subtitle: str
    sections: list[ArticleSection]

In [100]:
messages = [
    {"role": "user", "content": "tell me a bad time story about a unicorn"}
]

try:
    response = openai_client.responses.parse(
        model="gpt-4o-mini",
        input=messages,
        text_format=ArticleResponse,
        stream=True  # This doesn't work
    )
except Exception as e:
    print(f"Error: {type(e).__name__}: {e}")

Error: AttributeError: 'str' object has no attribute 'output'


In [101]:
messages = [
    {"role": "user", "content": "tell me a bad time story about a unicorn"}
]

with openai_client.responses.stream(
    model="gpt-4o-mini",
    input=messages,
    text_format=ArticleResponse,
) as stream:
    response = None
    for event in stream:
        if hasattr(event, 'delta'):
            print(event.delta, end='')
        if hasattr(event, 'response'):
            response = event.response

{"title":"The Unicorn's Unfortunate Adventure","subtitle":"A Tale of a Not-So-Perfect Day","sections":[{"header":"Once Upon a Time in Sparklewood","text":"In the enchanting land of Sparklewood, where rainbows danced in the sky and flowers sang melodies, lived a unicorn named Luma. With a shimmering coat and a mane that sparkled like stardust, Luma was adored by every creature in the forest."},{"header":"The Day Everything Went Wrong","text":"One sunny morning, Luma decided to explore beyond her familiar meadows. Full of excitement, she trotted past the Glimmering Stream and into the Misty Woods, a place whispered to be filled with surprises. However, as she ventured deeper, the sun vanished behind dark clouds, and a light drizzle began to fall."},{"header":"A Sticky Situation","text":"Luma, not familiar with the woods, tried to run back home, but instead found herself stuck in a muddy marsh! The more she struggled to free her hooves, the deeper she sank. What a transformation from a gr

In [102]:
story = response.output_parsed

print('# ' + story.title)
print(story.subtitle)
print()

for section in story.sections:
    print('## ' + section.header)
    print()
    print(section.text)
    print()
    print()

# The Unicorn's Unfortunate Adventure
A Tale of a Not-So-Perfect Day

## Once Upon a Time in Sparklewood

In the enchanting land of Sparklewood, where rainbows danced in the sky and flowers sang melodies, lived a unicorn named Luma. With a shimmering coat and a mane that sparkled like stardust, Luma was adored by every creature in the forest.


## The Day Everything Went Wrong

One sunny morning, Luma decided to explore beyond her familiar meadows. Full of excitement, she trotted past the Glimmering Stream and into the Misty Woods, a place whispered to be filled with surprises. However, as she ventured deeper, the sun vanished behind dark clouds, and a light drizzle began to fall.


## A Sticky Situation

Luma, not familiar with the woods, tried to run back home, but instead found herself stuck in a muddy marsh! The more she struggled to free her hooves, the deeper she sank. What a transformation from a graceful unicorn to a muddy mess!


## Unexpected Friends

Just as panic set in, a 

In [104]:
from jaxn import JSONParserHandler, StreamingJSONParser

In [105]:
from typing import Any, Dict

class ArticleResponseHandler(JSONParserHandler):

    def on_field_end(self, path: str, field_name: str, value: str, parsed_value: Any = None) -> None:
        if path == '':
            if field_name == 'title':
                print(f'# {value}')
            if field_name == 'subtitle':
                print(value)
        if path == '/sections' and field_name == 'header':
            print(f'\n\n## {value}\n')

    def on_value_chunk(self, path: str, field_name: str, chunk: str) -> None:
        if path == '/sections' and field_name == 'text':
            print(chunk, end='', flush=True)

In [106]:
handler = ArticleResponseHandler()
parser = StreamingJSONParser(handler=handler)

In [108]:
def llm_structured_stream(
    user_prompt,
    output_type,
    parser_handler=JSONParserHandler(),
    instructions=None,
    model="gpt-4o-mini",
):
    messages = []

    if instructions:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    parser = StreamingJSONParser(handler=parser_handler)

    with openai_client.responses.stream(
        model="gpt-4o-mini",
        input=messages,
        text_format=output_type,
    ) as stream:
        response = None
        for event in stream:
            if hasattr(event, 'delta'):
                parser.parse_incremental(event.delta)
            if hasattr(event, 'response'):
                response = event.response

    return response

In [109]:
instructions = "your task is to tell the user bad time stories"
user_prompt = "unicorn"

result = llm_structured_stream(
    instructions=instructions,
    user_prompt=user_prompt,
    output_type=ArticleResponse,
    parser_handler=ArticleResponseHandler(),
)

# The Dark Truth of the Unicorn
A Tale of Enchantment Gone Wrong


## A Glimpse of Beauty

Once upon a time, in a land filled with vibrant rainbows and luscious green pastures, a unicorn named Liora roamed freely. With her shimmering white coat and a spiraling horn that glowed in the sunlight, she was the pinnacle of beauty and grace. Villagers would travel from great distances just for a chance to catch a glimpse of her.

## The Curse of Envy

But not everything was as perfect as it seemed. A witch, envious of Liora's beauty, devised a plan to steal her magic. She wove a spell that would cloud the hearts of those who came near the unicorn, turning their adoration into insatiable greed.

## The Fall from Grace

On a fateful day, a group of villagers approached Liora with open hearts. However, under the witch's curse, their love morphed into jealousy. They plotted to capture the unicorn, believing that if they possessed her, they would gain her beauty and magic for themselves. One brave

In [110]:
import json

from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

print(f"Indexed {len(chunked_docs)} chunks from {len(files)} documents")

def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return prompt_template.format(
        question=question,
        context=context
    )


Indexed 385 chunks from 95 documents


In [111]:
from typing import Literal

class RAGResponse(BaseModel):
    """
    This model provides a structured answer with metadata about the response,
    including confidence, categorization, and follow-up suggestions.
    """

    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how certain the answer is")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user might want to ask")


In [112]:
class RAGResponseHandler(JSONParserHandler):
    def on_value_chunk(self, path: str, field_name: str, chunk: str) -> None:
        if path == '' and field_name == 'answer':
            print(chunk, end='', flush=True)

    def on_field_end(self, path: str, field_name: str, value: str, parsed_value: Any = None) -> None:
        if path == '' and field_name == 'answer_type':
            print('\nanswer type:', value)

    def on_array_item_end(self, path: str, field_name: str, item: Dict[str, Any] = None) -> None:
        if field_name == 'followup_questions':
            print('follow up question:', item)


In [113]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm_structured_stream(
        instructions=instructions,
        user_prompt=prompt,
        output_type=RAGResponse,
        parser_handler=RAGResponseHandler()
    )


In [114]:
response = rag('llm as a judge')

Using an LLM (Large Language Model) as a judge involves creating a system that evaluates responses based on specific criteria. The tutorial outlines two primary ways to implement an LLM as a judge:

1. **Reference-based Evaluation**: This method compares new responses against approved, reference answers. It is particularly useful for regression testing where a "ground truth" is available.

2. **Open-ended Evaluation**: This approach allows for the evaluation of responses based on custom criteria when no definitive reference is available.

The main steps to create and evaluate an LLM judge include:
- **Creating an evaluation dataset**: This consists of questions, target responses, new responses, and manual labels indicating correctness.
- **Designing an LLM evaluator prompt**: This can involve custom templates that guide the LLM on how to evaluate the responses.
- **Comparing evaluations**: The judgments made by the LLM can be compared against manually assigned labels to assess its accu

In [115]:
response

ParsedResponse[RAGResponse](id='resp_063d87ba392d9e5a00697e0d0914788196b7f09cfe8ad824c0', created_at=1769868553.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-mini-2024-07-18', object='response', output=[ParsedResponseOutputMessage[RAGResponse](id='msg_063d87ba392d9e5a00697e0d09b6548196af71004d5eede95c', content=[ParsedResponseOutputText[RAGResponse](annotations=[], text='{"answer":"Using an LLM (Large Language Model) as a judge involves creating a system that evaluates responses based on specific criteria. The tutorial outlines two primary ways to implement an LLM as a judge:\\n\\n1. **Reference-based Evaluation**: This method compares new responses against approved, reference answers. It is particularly useful for regression testing where a \\"ground truth\\" is available.\\n\\n2. **Open-ended Evaluation**: This approach allows for the evaluation of responses based on custom criteria when no definitive reference is available.\\n\\nThe main steps